In [4]:
import torch
import open_clip
from peft import LoraConfig, get_peft_model

# 1. Load the model as you did before
model, _, preprocess = open_clip.create_model_and_transforms('ViT-g-14', pretrained='laion2b_s34b_b88k')

# 2. Updated LoRA Config for OpenCLIP architecture
# OpenCLIP ViT models usually name their attention projections "in_proj" 
# (which handles Q, K, and V) and "out_proj".
config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["in_proj", "out_proj"], # Correct names for OpenCLIP ViT
    lora_dropout=0.05, 
    bias="none"
)

# 3. Apply LoRA to the visual part
model.visual = get_peft_model(model.visual, config)
model.cuda()

print("Success! Trainable parameters:")
model.visual.print_trainable_parameters()

Success! Trainable parameters:
trainable params: 1,802,240 || all params: 1,014,447,872 || trainable%: 0.1777


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset
import open_clip
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader

# --- CONFIGURATION ---
MODEL_NAME = 'ViT-g-14'
PRETRAINED = 'laion2b_s34b_b88k'
BATCH_SIZE = 8  # Reduced for ViT-g-14 VRAM safety
EPOCHS = 3
LEARNING_RATE = 5e-5
POST_IMP_LABEL = 20 # Style ID for Post-Impressionism in huggan/wikiart

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Load and Filter Dataset (WikiArt)
    print("Downloading and filtering WikiArt dataset...")
    # This might take a moment to download (~5-10GB)
    dataset = load_dataset("huggan/wikiart", split="train")
    
    # Filter for only Post-Impressionist works
    post_imp_ds = dataset.filter(lambda x: x['style'] == POST_IMP_LABEL)
    print(f"Found {len(post_imp_ds)} Post-Impressionist artworks.")

    # 2. Setup Model & LoRA
    model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
    
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["in_proj", "out_proj"], # Fixed for OpenCLIP
        lora_dropout=0.05,
        bias="none"
    )
    
    model.visual = get_peft_model(model.visual, config)
    
    # We add a small head to learn Artist characteristics within Post-Impressionism
    # huggan/wikiart has 129 artists
    model.visual.head = nn.Linear(model.visual.output_dim, 129).to(device)
    model.to(device)

    # 3. Data Prep
    def transform(examples):
        # Apply CLIP preprocessing to the images
        examples["pixel_values"] = [preprocess(image) for image in examples["image"]]
        return examples

    # Set the transform on the fly
    post_imp_ds.set_transform(transform)
    
    # Use a standard PyTorch DataLoader
    def collate_fn(batch):
        return {
            'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
            'labels': torch.tensor([x['artist'] for x in batch])
        }

    train_loader = DataLoader(post_imp_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    # 4. Training Loop
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    model.train()
    print("Starting Fine-tuning on Post-Impressionism...")
    for epoch in range(EPOCHS):
        for i, batch in enumerate(train_loader):
            inputs, labels = batch['pixel_values'].to(device), batch['labels'].to(device)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                features = model.encode_image(inputs)
                logits = model.visual.head(features)
                loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            
            if i % 50 == 0:
                print(f"Epoch {epoch+1} | Batch {i} | Loss: {loss.item():.4f}")

    # 5. Save the 'Artistic Brain'
    torch.save(model.visual.state_dict(), "vit_g_post_impressionism_lora.pt")
    print("Fine-tuning complete! Weights saved as 'vit_g_post_impressionism_lora.pt'")

if __name__ == "__main__":
    main()

Using device: cuda


OSError: Not enough disk space. Needed: 66.20 GiB (download: 31.42 GiB, generated: 34.79 GiB, post-processed: Unknown size)